In [1]:
pip install noisereduce

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import pandas as pd
import librosa
import soundfile as sf
import noisereduce as nr

# ============================================================
# CONFIGURATION
# ============================================================

TIMIT_DIR = Path("../data/raw/timit")

PROCESSED_DIR = Path("../data/processed/timit")

TARGET_SAMPLE_RATE = 16000

NOISE_REDUCTION_STRENGTH = 0.7

# ============================================================
# FIND AUDIO FILES
# ============================================================

def find_audio_files(timit_dir=TIMIT_DIR):

    wav_files = list(timit_dir.rglob("*.WAV.wav"))

    return sorted(wav_files)


# ============================================================
# EXTRACT SPEAKER ID
# ============================================================

def get_speaker_id(audio_path):

    """
    TIMIT structure:

    TIMIT/
        TRAIN/
            DR1/
                FCJF0/
                    SA1.WAV

    Speaker ID = FCJF0
    """

    return audio_path.parent.name


# ============================================================
# LOAD AUDIO
# ============================================================

def load_audio(audio_path):

    audio, sr = librosa.load(
        audio_path,
        sr=TARGET_SAMPLE_RATE,
        mono=True
    )

    return audio, sr

def reduce_noise(audio, sr=TARGET_SAMPLE_RATE):

    # Avoid processing empty or extremely short audio.
    if len(audio) == 0:
        return audio

    # Apply spectral noise reduction.
    # stationary=False is suitable for noise that changes
    # over time. It does not require a separate noise recording.
    cleaned_audio = nr.reduce_noise(
        y=audio,
        sr=sr,
        stationary=False,
        prop_decrease=NOISE_REDUCTION_STRENGTH
    )

    return cleaned_audio

# ============================================================
# TRIM SILENCE
# ============================================================

def trim_silence(audio):

    trimmed_audio, _ = librosa.effects.trim(
        audio,
        top_db=30
    )

    return trimmed_audio


# ============================================================
# NORMALIZE AUDIO
# ============================================================

def normalize_audio(audio):

    max_amplitude = abs(audio).max()

    if max_amplitude == 0:

        return audio

    return audio / max_amplitude


# ============================================================
# READ TIMIT TRANSCRIPT
# ============================================================

def get_transcript(audio_path):

    transcript_file = audio_path.with_suffix(".TXT")

    if not transcript_file.exists():

        return ""

    with open(
        transcript_file,
        "r",
        encoding="utf-8"
    ) as f:

        line = f.readline().strip()

    parts = line.split()

    if len(parts) < 3:

        return ""

    # TIMIT format:
    #
    # start_time end_time transcript

    transcript = " ".join(parts[2:])

    return transcript


# ============================================================
# SAVE AUDIO
# ============================================================

def save_audio(
    audio,
    output_path
):

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    sf.write(
        output_path,
        audio,
        TARGET_SAMPLE_RATE
    )


# ============================================================
# PROCESS DATASET
# ============================================================

def process_timit():

    audio_files = find_audio_files()

    print(
        f"Found {len(audio_files)} audio files."
    )

    metadata = []

    for i, audio_path in enumerate(audio_files):

        try:

            audio, sr = load_audio(
                audio_path
            )
            
            audio = reduce_noise(
                audio,
                sr=sr
            )
            audio = trim_silence(
                audio
            )

            audio = normalize_audio(
                audio
            )

            speaker_id = get_speaker_id(
                audio_path
            )

            transcript = get_transcript(
                audio_path
            )

            relative_path = (
                audio_path.relative_to(
                    TIMIT_DIR
                )
            )

            output_path = (
                PROCESSED_DIR /
                relative_path
            )

            save_audio(
                audio,
                output_path
            )

            duration = (
                len(audio) /
                TARGET_SAMPLE_RATE
            )

            metadata.append({

                "audio_path": str(
                    output_path
                ),

                "speaker_id": speaker_id,

                "transcript": transcript,

                "duration": duration,

                "sample_rate": sr

            })

        except Exception as e:

            print(
                f"Error processing "
                f"{audio_path}: {e}"
            )

    metadata_df = pd.DataFrame(
        metadata
    )

    metadata_path = (
        PROCESSED_DIR /
        "metadata.csv"
    )

    metadata_df.to_csv(
        metadata_path,
        index=False
    )

    print(
        f"\nProcessed "
        f"{len(metadata_df)} files."
    )

    print(
        f"Metadata saved to "
        f"{metadata_path}"
    )


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    process_timit()

/Users/melaniequ/vcfad-tts/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 5670 audio files.

Processed 5670 files.
Metadata saved to ../data/processed/timit/metadata.csv
